# 🧹 Telco Customer Churn — Data Cleaning

**Dataset:** `WA_Fn-UseC_-Telco-Customer-Churn.csv`  
**Goal:** Clean and prepare the raw dataset for EDA and modelling.

### 🗂️ Cleaning Steps
1. Load & inspect raw data
2. Fix `TotalCharges` — convert from object → float (11 whitespace entries)
3. Handle missing values created after type conversion
4. Convert `SeniorCitizen` from int (0/1) → readable Yes/No
5. Simplify multi-value columns (`No phone service`, `No internet service` → `No`)
6. Convert `Churn` to binary int (0/1) for modelling
7. Standardise column names (snake_case)
8. Remove duplicates (sanity check)
9. Validate & export cleaned data

## 1️⃣ Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('✅ Libraries loaded successfully')

## 2️⃣ Load Raw Data

In [ ]:
RAW_PATH = '../data/WA_Fn-UseC_-Telco-Customer-Churn.csv'

df = pd.read_csv(RAW_PATH)

print(f'📊 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

## 3️⃣ Initial Inspection

In [ ]:
print('🔍 Data Types & Non-Null Counts')
print('=' * 45)
df.info()

In [ ]:
print('📈 Numerical Summary')
df.describe()

In [ ]:
print('🔎 Null / Missing Values per Column')
print('=' * 45)
null_summary = pd.DataFrame({
    'Null Count'  : df.isnull().sum(),
    'Null %'      : (df.isnull().sum() / len(df) * 100).round(2),
    'Dtype'       : df.dtypes
})
if null_summary['Null Count'].sum() > 0:
    print(null_summary[null_summary['Null Count'] > 0])
else:
    print('✅ No null values found in raw data')

In [ ]:
print(f'🔁 Duplicate rows: {df.duplicated().sum()}')

## 4️⃣ Fix `TotalCharges` Column

> **Issue:** `TotalCharges` is stored as `object`. It contains **11 whitespace-only** (`' '`) entries instead of NaN — these are customers with `tenure = 0` (new customers with no charges yet).

In [ ]:
# Identify whitespace entries
whitespace_mask = df['TotalCharges'].str.strip() == ''
print(f'⚠️  Whitespace entries in TotalCharges: {whitespace_mask.sum()}')
print('\nRows with whitespace TotalCharges:')
df[whitespace_mask][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']]

In [ ]:
# Replace whitespace with NaN, then convert to float
df['TotalCharges'] = df['TotalCharges'].str.strip().replace('', np.nan).astype(float)

print(f'✅ TotalCharges dtype after fix: {df["TotalCharges"].dtype}')
print(f'   Missing values created: {df["TotalCharges"].isnull().sum()}')

## 5️⃣ Handle Missing Values in `TotalCharges`

> These 11 customers have `tenure = 0`, meaning they just joined. Their `TotalCharges` should logically be **0.0** (no billing cycle completed yet).

In [ ]:
# Verify: all missing TotalCharges have tenure == 0
missing_mask = df['TotalCharges'].isnull()
print('Tenure values for rows with missing TotalCharges:')
print(df[missing_mask]['tenure'].value_counts())

# Fill with 0.0 — these customers have completed no billing cycle
df['TotalCharges'] = df['TotalCharges'].fillna(0.0)

print(f'\n✅ Missing values after fill: {df["TotalCharges"].isnull().sum()}')

## 6️⃣ Convert `SeniorCitizen` — 0/1 → Yes/No

In [ ]:
print('Before:', df['SeniorCitizen'].value_counts().to_dict())

df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})

print('After :', df['SeniorCitizen'].value_counts().to_dict())
print('✅ SeniorCitizen converted to Yes/No')

## 7️⃣ Simplify Multi-Service Columns

> Columns like `MultipleLines`, `OnlineSecurity`, `OnlineBackup`, etc. contain values like `'No phone service'` and `'No internet service'`.  
> These are just `'No'` — the extra context is already captured in `PhoneService` / `InternetService`. We simplify them.

In [ ]:
# Columns with 'No phone service'
phone_cols = ['MultipleLines']

# Columns with 'No internet service'
internet_cols = [
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies'
]

# Replace 'No phone service' → 'No'
for col in phone_cols:
    before = (df[col] == 'No phone service').sum()
    df[col] = df[col].replace('No phone service', 'No')
    print(f'  {col}: replaced {before} "No phone service" → "No"')

print()

# Replace 'No internet service' → 'No'
for col in internet_cols:
    before = (df[col] == 'No internet service').sum()
    df[col] = df[col].replace('No internet service', 'No')
    print(f'  {col}: replaced {before} "No internet service" → "No"')

print('\n✅ Multi-service columns simplified')

## 8️⃣ Standardise Column Names → snake_case

In [ ]:
import re

def to_snake_case(name):
    s1 = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', name)
    return re.sub('([a-z0-9])([A-Z])', r'\1_\2', s1).lower()

old_cols = df.columns.tolist()
df.columns = [to_snake_case(col) for col in df.columns]
new_cols = df.columns.tolist()

print('Column name mapping:')
for old, new in zip(old_cols, new_cols):
    marker = '✏️ ' if old != new else '   '
    print(f'  {marker} {old:22s} → {new}')

## 9️⃣ Encode Target Column `churn` → 0 / 1

In [ ]:
print('Before:', df['churn'].value_counts().to_dict())

df['churn'] = df['churn'].map({'No': 0, 'Yes': 1})

print('After :', df['churn'].value_counts().to_dict())
churn_rate = df['churn'].mean() * 100
print(f'\n📊 Churn Rate: {churn_rate:.1f}%')
print('✅ Churn encoded: No → 0, Yes → 1')

## 🔟 Remove Duplicate Rows (Final Check)

In [ ]:
dupes_before = df.duplicated().sum()
df = df.drop_duplicates()
print(f'Duplicate rows removed: {dupes_before}')
print(f'✅ Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

## 1️⃣1️⃣ Final Validation

In [ ]:
print('=' * 55)
print('           ✅ CLEANED DATASET SUMMARY')
print('=' * 55)
print(f'  Rows          : {df.shape[0]:,}')
print(f'  Columns       : {df.shape[1]}')
print(f'  Missing Values: {df.isnull().sum().sum()}')
print(f'  Duplicates    : {df.duplicated().sum()}')
print(f'  Churn Rate    : {df["churn"].mean()*100:.1f}%')
print('=' * 55)
print()
print('🔢 Data Types:')
print(df.dtypes)

In [ ]:
# Final null check per column
null_after = df.isnull().sum()
if null_after.sum() == 0:
    print('✅ Zero missing values — data is clean!')
else:
    print('⚠️  Remaining missing values:')
    print(null_after[null_after > 0])

In [ ]:
df.head(10)

## 1️⃣2️⃣ Quick Visualisation — Class Balance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Churn Distribution in Cleaned Dataset', fontsize=14, fontweight='bold')

colors = ['#6366f1', '#f43f5e']

# Bar chart
churn_counts = df['churn'].value_counts()
axes[0].bar(['No Churn (0)', 'Churn (1)'], churn_counts, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Churn Count')
axes[0].set_ylabel('Number of Customers')
for i, v in enumerate(churn_counts):
    axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(
    churn_counts,
    labels=['No Churn', 'Churn'],
    autopct='%1.1f%%',
    colors=colors,
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[1].set_title('Churn Split (%)')

plt.tight_layout()
plt.savefig('../data/churn_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Plot saved to data/churn_distribution.png')

## 1️⃣3️⃣ Export Cleaned Data

In [ ]:
CLEANED_PATH = '../data/telco_churn_cleaned.csv'

df.to_csv(CLEANED_PATH, index=False)

print(f'✅ Cleaned dataset saved to: {CLEANED_PATH}')
print(f'   Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

---

## 📋 Cleaning Summary

| Step | Action | Details |
|------|--------|---------|
| 1 | **Fix TotalCharges** | Converted `object` → `float`; 11 whitespace entries found |
| 2 | **Fill Missing Values** | 11 NaNs in `total_charges` filled with `0.0` (tenure = 0 customers) |
| 3 | **SeniorCitizen** | Mapped `0/1` → `No/Yes` for readability |
| 4 | **Simplify Services** | Replaced `'No phone service'` & `'No internet service'` → `'No'` |
| 5 | **Column Names** | Standardised to `snake_case` |
| 6 | **Encode Target** | `Churn`: `No → 0`, `Yes → 1` |
| 7 | **Duplicates** | 0 duplicates found |

**Output:** `data/telco_churn_cleaned.csv` — ready for EDA & modelling 🚀